# 14_train_evaluate_supervised_ML_models

Run supervised ML training/evaluation across feature combinations, split types, and model choices.

## 1) Environment Setup


In [ ]:
from pathlib import Path
import sys

# Section 1: Resolve repo root from either project root or notebooks/ cwd.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

# Section 2: Add repo and src roots to sys.path for imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


## 2) Imports


In [ ]:
from pprint import pprint
from project_config.feature_registry import COMBI_ML_FEATURE_SETS
from project_config.variables import address_dict


## 3) Configure ML Inputs


In [ ]:
import abcode.steps.train_evaluate_supervised_ml_models as ml_step
default_user_inputs = ml_step.default_user_inputs
user_inputs = default_user_inputs()

# Edit these for your run
user_inputs['root_key'] = 'examples' # 'biostream-developability-data'
user_inputs['data_fbase'] = user_inputs['root_key']
user_inputs['data_subfolder'] = 'opensource' #
user_inputs['csv_suffix'] = '_acsins_xgboost' #
default_dataset_fname = user_inputs['data_subfolder'] + '.csv'
user_inputs['dataset_fname'] = default_dataset_fname

# Experimental data parent folder: address_dict[root_key] / "expdata" / data_subfolder
expdata_dir = (repo_root / Path(address_dict[user_inputs['root_key']]) / 'expdata' / user_inputs['data_subfolder']).resolve()
user_inputs['input_filename_prefix'] = user_inputs['data_subfolder'] + '_' # ''
user_inputs['sequence_base'] = None
user_inputs['target_col'] = ['acsins_dLmax_ph7.4_avg'] # ['hic_rt_avg', 'sec_%monomer_avg', 'acsins_dLmax_ph7.4_avg', 'tm1_nanodsf_avg']
user_inputs['classification_or_regression'] = 'regression'

# specify splits
user_inputs['split_type_list'] = ['random']
user_inputs['k_folds'] = 5
user_inputs['random_kfold_repeats'] = 1  # >1 enables repeated random k-fold from scratch
user_inputs['random_split_col'] = f'fold_random_{user_inputs["k_folds"]}'
user_inputs['mutres_split_col'] = f'fold_mutres-modulo_{user_inputs["k_folds"]}'
user_inputs['contiguous_split_col'] = f'fold_contiguous_{user_inputs["k_folds"]}'  # optional precomputed column
user_inputs['custom_split_col'] = 'fold_custom'
user_inputs['custom_test_value'] = 1
# specify custom dataset name
user_inputs['custom_test_dataset_fname'] = None #
user_inputs['custom_test_data_subfolder'] = user_inputs['data_subfolder']
user_inputs['custom_input_filename_prefix'] = None if user_inputs['custom_test_dataset_fname'] is None else user_inputs['custom_test_dataset_fname'].split('.')[0] # ''

# specify featureset combinations to test
BEST_FEATURES = {
        'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    'physicochemical_seq_all': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac'],
}
# user_inputs['feature_combinations_dict'] = COMBI_ML_FEATURE_SETS
user_inputs['feature_combinations_dict'] = {
    # 'vh_esm2_seq_embeddings': ['vh_esm2-650m_mean_pooled-33'],
    # 'cdrh3_esm2_seq_embeddings': ['cdrh3_esm2-650m_mean_pooled-33'],
    # 'vh_cdrh3_esm2_seq_embeddings': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_mean_pooled-33'],
    # 'vh_esm2_seq_embeddings_cdrh_esm2_pll': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_PLL-wt', ],
    # 'vh_esm2_seq_embeddings_svd20_cdrh_esm2_pll': ['vh_esm2-650m_mean_pooled-33_svd20', 'cdrh3_esm2-650m_PLL-wt', ],
    # 'vh_vl_georgiev_mean_pooled': ['vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled']
    # 'vh_esm2_seq_embeddings_cdrh_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'vh_esm2_seq_embeddings_svd20_cdrh_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd20', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'vhvlcdrh3_esm2_seq_embeddings_svd50_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd50', 'vl_esm2-650m_mean_pooled-33_svd50', 'cdrh3_esm2-650m_mean_pooled-33_svd50', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    #     'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
    # 'physicochemical_seq_all': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac'],
    # 'vhvlcdrh3_esm2_seq_embeddings_svd80_vhvlcdrh3_esm2_pll_georgiev_mean_pooled': ['vh_esm2-650m_mean_pooled-33_svd80', 'vl_esm2-650m_mean_pooled-33_svd80', 'cdrh3_esm2-650m_mean_pooled-33_svd80', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt', 'vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
        'physicochemical_seq_all_esm2_seq_embeddings_svd50_georgiev': ['cdr_length', 'vh_aac', 'vh_aaindex1', 'vh_ctdc', 'vh_ctdc', 'vl_aac', 'vl_aaindex1','vl_ctdc', 'vl_ctdc', 'cdr_length', 'cdrh3_length', 'cdrh3_aaindex1', 'cdrh3_aac','vh_esm2-650m_mean_pooled-33_svd50', 'vl_esm2-650m_mean_pooled-33_svd50', 'cdrh3_esm2-650m_mean_pooled-33_svd50', 'vh_esm2-650m_PLL-wt', 'vl_esm2-650m_PLL-wt', 'cdrh3_esm2-650m_PLL-wt','vh_georgiev_mean_pooled', 'vl_georgiev_mean_pooled'],
}


# specify models
user_inputs['model_list'] =  ['xgboost']# ['ridge', 'mlp_pytorch'] # ['ridge']

# run parameters
# hyperparameter tuning
user_inputs['hyperparameter_mode'] = 'optuna_search' # None # 'preset_search' # 'default', 'ntrain_preset',
user_inputs['tuning_metric'] = 'spearman'
user_inputs['tuning_n_trials'] = 20
user_inputs['summary_source_mode'] = 'all_saved_rows'  # 'run_cache' or 'all_saved_rows'
user_inputs['summary_metric_mode'] = 'average'  # 'average' or 'pooled'

# save parameters
user_inputs['save_trained_models'] = False
user_inputs['save_predictions'] = True
user_inputs['train_full_data_model'] = False
user_inputs['featurecombi_model_pair_to_extract_coefficients_for'] = None # [('onehot', 'ridge')]
user_inputs['show_progress'] = True
pprint(user_inputs)


## 4) Run Training And Evaluation


In [ ]:
import importlib

import abcode.tools.ml.model_registry as model_registry
import abcode.tools.ml.hyperparameter_tuning as hyperparameter_tuning
import abcode.tools.ml.workflow as workflow
import abcode.steps.train_evaluate_supervised_ml_models as ml_step

importlib.reload(model_registry)
importlib.reload(hyperparameter_tuning)
importlib.reload(workflow)
ml_step = importlib.reload(ml_step)

result = ml_step.train_evaluate_supervised_ml_models(user_inputs)
pprint(result)


## 6) Visualize ML Results


In [ ]:
from pprint import pprint

import importlib

import abcode.steps.visualize_ml_results as ml_viz_step
ml_viz_step = importlib.reload(ml_viz_step)

visualize_ml_results = ml_viz_step.visualize_ml_results
default_viz_inputs = ml_viz_step.default_user_inputs

viz_inputs = default_viz_inputs()
viz_inputs['data_fbase'] = user_inputs.get('data_fbase', 'examples')
viz_inputs['data_subfolder'] = user_inputs.get('data_subfolder', '')
viz_inputs['metrics_summary_fname'] = f"{user_inputs['classification_or_regression']}_metrics_summary{user_inputs.get('csv_suffix', '')}.csv"
viz_inputs['model_name_list'] = ['best'] # ['xgboost', 'ridge']
viz_inputs['metric_col'] = ['test_spearman']  # string or list
viz_inputs['target_col_list'] = user_inputs.get('target_col', []) if isinstance(user_inputs.get('target_col', []), list) else [user_inputs.get('target_col')]
viz_inputs['split_type_list'] = []  # [] -> iterate all split types in summary CSV
viz_inputs['feature_label_list'] = list(user_inputs['feature_combinations_dict'].keys())
viz_inputs['show_model_type'] = False  # True -> marker shape based on model type
viz_inputs['save_figure'] = True
viz_inputs['show_figure'] = True
viz_inputs['figure_output_dir'] = ''  # default: summary csv folder
viz_inputs['figure_fname'] = ''       # default auto name
viz_inputs['x_limits'] = [0, None]  # e.g. [0, 10000]; None/[] uses matplotlib defaults
viz_inputs['y_limits'] = [0, 1]      # e.g. [0.0, 1.0]; None/[] uses matplotlib defaults

# Optional override:
viz_inputs['metrics_summary_csv_path'] = '/Users/charmainechia/Documents/projects/MUTAGENESIS-DATA-BENCHMARKS/ml/output/D7PM05_CLYGR_Somermeyer_2022/regression_metrics_summary_merged-0-1.csv'

viz_result = visualize_ml_results(viz_inputs)
pprint(viz_result)
